In [98]:
import GtoTmodel as GtoTmodel
import torch
import torch.nn as nn
import torch.optim as optim

In [99]:
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 2  # Number of transformer layers
dropout = 0.1  # Dropout rate

graph_colomns=5
num_components=5
batch_size = 6 # Batch size

graph_input_dim = 10  # Number of colomns in the graph
text_vocab_size = 26  # Vocabulary size for text

Importing the model

In [100]:
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

c:\Users\MSI\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [101]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [102]:
learning_rate = 0.001
num_epochs = 10000

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Sample trainig set

In [103]:
num_ciruits = 100
graph_dataset =  torch.randn(num_ciruits,graph_colomns, graph_input_dim )
text_dataset = torch.randint(0, num_components, (num_ciruits,graph_colomns))

graph_dataset = graph_dataset.to(device)
text_dataset = text_dataset.to(device)


In [104]:
graph_dataset[0]

tensor([[ 0.8929,  1.1544, -0.5669, -1.0311,  0.6939,  0.9049,  0.0129,  0.4812,
          0.6557, -1.3978],
        [ 0.4593,  0.3388,  0.1833,  0.0690,  0.7348,  0.5058, -0.9420,  1.8228,
          0.7196,  0.8772],
        [ 1.1554,  0.5115,  1.7552, -1.5965,  0.8164, -1.0756, -0.5344, -0.0694,
         -1.1041,  0.0204],
        [-0.2714,  1.0326,  1.2683, -0.8144, -1.2171,  1.1222,  0.4555, -0.5960,
          0.2080, -0.5368],
        [ 1.6482, -0.7132, -0.0159, -0.7560, -0.4343, -0.1334, -0.5445,  1.7045,
          0.0766,  0.2317]], device='cuda:0')

Split data in to test and train set

In [105]:
from sklearn.model_selection import train_test_split

graph_train, graph_test, text_train, text_test = train_test_split(graph_dataset, text_dataset, test_size=0.2)



In [106]:
# Assuming train_graph_data and train_text_input are the training datasets
train_graph_data = torch.randn(100, graph_rows, graph_input_dim)  # Example training graph data
train_text_input = torch.randint(0, text_vocab_size, (100, num_components))  # Example training text input

# Training loop
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode

    for i in range(len(train_graph_data)):
        graph_data = train_graph_data[i].unsqueeze(0)  # Get graph data by index and add batch dimension
        text_input = train_text_input[i].unsqueeze(0)  # Get text input by index and add batch dimension

        # Forward pass
        output = model(graph_data, text_input[:, :-1])  # Exclude the last token for input
        output = output.reshape(-1, text_vocab_size)  # Reshape for loss calculation

        target = text_input[:, 1:].reshape(-1)  # Exclude the first token for target

        # Compute loss
        loss = criterion(output, target)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Print loss for the epoch
    if (epoch + 1) % 100 == 0:  # Print every 100 epochs
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

NameError: name 'graph_rows' is not defined

In [109]:
# Training loop with relative graph input
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode

    # Forward pass
    for i in range(0, len(graph_train), batch_size):
        graph_data = graph_train[i:i+batch_size]
        text = text_train[i:i+batch_size]
        text_input = text[:, :-1]
        target = text[:, -1].reshape(-1)  # Use the last token as the target
        relative_graph_input = graph_data[:, 1:]  # Use relative graph input (excluding the first column)
        
        output = model(relative_graph_input, text_input)  # Pass relative graph input and text input to the model
        print(output.shape)
        output = output.reshape(-1, text_vocab_size)  # Reshape for loss calculation
        print(output.shape)
        print(target.shape)
        # Compute loss
        loss = criterion(output, target)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Print loss for the epoch
    if (epoch + 1) % 100 == 0:  # Print every 100 epochs
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

torch.Size([6, 4, 26])
torch.Size([24, 26])
torch.Size([6])


ValueError: Expected input batch_size (24) to match target batch_size (6).

In [ ]:
# Training loop
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode

    # Forward pass
    for i in range(0, len(graph_train), batch_size):
        graph_data = graph_train[i:i+batch_size]
        text = text_train[i:i+batch_size]
        text_input = text[:, :-1]
        target = text[:,-2:-1].reshape(-1)
        output = model(graph_data[:, :-1], text_input)  # Exclude the last token for input
        print(output.shape)
        print(output)
        output = output.reshape(-1, text_vocab_size)  # Reshape for loss calculation
        print(output.shape)
        
        # Compute loss
        loss = criterion(output, target)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    # Print loss for the epoch
    if (epoch + 1) % 100 == 0:  # Print every 100 epochs
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

torch.Size([5, 4, 26])
tensor([[[-1.5100,  1.2659,  0.1640,  0.8427,  0.8716,  0.0361,  0.3355,
           0.4042, -0.2327,  0.2317,  0.0522,  0.0401, -0.1653, -0.5668,
          -0.5292,  0.1564,  1.2365, -0.1257, -0.2466,  0.2826, -0.7322,
          -0.0211, -0.8385,  0.2999,  0.7142,  0.3439],
         [-1.3764,  1.7454,  0.4456, -0.1342,  1.1913, -0.0195,  0.9055,
           0.8508, -1.0829,  1.1190,  0.4547, -0.2169,  0.0645, -0.2904,
          -0.7012, -0.2384,  1.1285, -0.1103, -0.1722, -1.1213, -0.0603,
          -0.7715, -0.6441,  0.2725,  0.0658,  0.0917],
         [-2.2379,  0.6691,  0.5675,  0.0804,  0.1499, -0.2961,  0.5484,
           0.8181, -0.8676,  0.6981,  0.2726, -0.1532,  0.1385,  0.3542,
          -0.9121, -0.5441,  0.0099, -0.3789, -0.3120, -1.3480,  0.5090,
          -0.6460, -1.0713,  1.0479,  0.1097, -0.0255],
         [-1.8912,  0.6884,  0.2930,  0.5582,  0.4600,  0.0543, -0.3521,
           0.5985, -1.2012,  0.6942,  0.6957, -0.4297, -0.7793,  0.4707,
      

ValueError: Expected input batch_size (20) to match target batch_size (5).

In [ ]:
# Forward pass
output = model(graph_data[:, :-1], text_input)  # Exclude the last token for input
output = output.reshape(-1, text_vocab_size)  # Reshape for loss calculation

# Compute loss
loss = criterion(output, target)
print(f"Loss: {loss.item():.4f}")